# Espinosa et al. (2017) on CRCNS bf-2

Paper: Espinosa N, Alonso A, Morales C, Espinosa P, Chávez AE, Fuentealba P (2017). Basal Forebrain
Gating by Somatostatin Neurons Drives Prefrontal Cortical Activity. Cereb Cortex 29:42-53,
doi:10.1093/cercor/bhx302.

Dataset: CRCNS bf-2, "Basal forebrain extracellular recordings of anesthetized mice with somatostatin
positive neurons optogenetically inhibited", doi:10.6080/K0MG7MQW, read from the public bucket
`crcnsarchive` under `bf-2/live/`. CRCNS data sets are not versioned, so the version is recorded as
`unversioned` (docs/archives.md, CRCNS); the paper cites no version, and there is only the one
published copy, so it is the one used. The files read and their MD5 sums from the data set's
`checksums.md5` (read 2026-11-09) are printed below, so a later change can be detected. The data
descriptor is version 0.6 (Dec 12, 2018).

Claims: `claims/espinosa-2017.yaml`, written from the version of record with its supplement and
checked by a second reader.

| Claim | Tested here |
|---|---|
| supp1-nphr-in-somatostatin-cells | Blocked: data missing (histology) |
| supp2-nphr-silences-som-cells | Blocked: data missing (in vitro) |
| supp3-two-ventral-pallidum-cell-types | Blocked: data missing (in vitro) |
| fig1-bf-response-classes | Tested |
| fig1d-excited-cells-fire-faster | Tested |
| fig1d-discharge-by-class | Tested |
| fig1e-excited-slower-kinetics | Tested |
| supp4-tail-pinch-response-patterns | Tested |
| fig2c, supp6, fig2d, supp7c, fig2e, fig3a, fig3b, supp9 (3 claims) | Blocked: data missing (prefrontal units) |
| fig4bcd, supp10, fig4g, supp11a, fig4h, suppt2 | Blocked: data missing (prefrontal LFP) |
| fig5c-light-path-needed | Blocked: data missing (light-blocked sessions) |
| fig5d-nphr-locomotion, supp14-moving-no-difference | Blocked: metadata insufficient (no laser timing for the NpHR- videos, no mouse identity for the per-train traces) |

The data set holds the ventral pallidum (basal forebrain) optrode recordings of Figure 1 and
Supplementary Figure 4, medial septum recordings for a later paper, and open-field videos. It holds
no prefrontal recordings, no in vitro or histology data, and no sessions with the light path blocked.

In [ ]:
import collections
import resource
import time
from concurrent.futures import ProcessPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

import bf2

T0 = time.time()
for k, v in bf2.MD5.items():
    print(f"{v}  {k}")
print("dataset:", bf2.DATASET)

## Loading

Each archive is a tar.gz file, streamed once from start to end; only the unit tables
(`data_correlation.mat`) and the light and pinch timing of each recording (`recxx_lan500.mat`, without
its LFP array) are kept. The two open-field archives are streamed for their file lists and small
.mat files. The six streams run in parallel.

In [ ]:
jobs = [("ephys", a) for a in bf2.VP_ANIMALS] + [
    ("beh", "Behavioral_tests/OPEN_FIELD/VP.tar.gz"), ("beh", "Behavioral_tests/OPEN_FIELD/Control.tar.gz")]


def load(job):
    kind, what = job
    return bf2.stream_animal(what) if kind == "ephys" else bf2.stream_behavior(what)


with ProcessPoolExecutor(6) as ex:
    loaded = list(ex.map(load, jobs))
animals = loaded[:4]
beh_vp, beh_ctl = loaded[4], loaded[5]
print(f"loaded in {time.time() - T0:.0f} s")

## Data check

In [ ]:
units, unit_animal = [], []
lan = {}
for r in animals:
    for rn, rec in r["recs"].items():
        lan[(r["animal"], rn.replace("rec", "reg"))] = rec["lan"]
    for u in r["units"]:
        units.append(u)
        unit_animal.append(r["animal"])
unit_animal = np.array(unit_animal)
print("header of data_correlation.mat:", animals[0]["header"])
print(f"animals {len(animals)}, recordings {len(lan)}, units {len(units)} (paper: 4 animals, 330 units; "
      "descriptor: 20 recordings, 330 neurons)")
for r in animals:
    cnt = collections.Counter(u["rec"] for u in r["units"])
    print(f"\n{r['animal']}: bad channels {r['bad_channel']}")
    for rn, rec in sorted(r["recs"].items()):
        L = rec["lan"]
        on, off = bf2.onsets_from_ttl(L["ttl"][:, 0], L["srate"])
        line = (f"  {rn} {rec['rhd']}: srate {L['srate']:.0f} Hz, {L['pnts'] / L['srate']:.1f} s, LFP array "
                f"{L['data_shape']}, laser TTL {len(on)} pulses of {np.median(off - on):.3f} s every "
                f"{np.median(np.diff(on)):.2f} s, good-signal fraction {L['selected'].mean():.3f}, "
                f"units {cnt[rn.replace('rec', 'reg')]}")
        line += f"; LAN.ttl rows {L['ttl'].shape[1]}, LAN fields {[k for k in ('RT', 'RTa', 'RTb') if k in L]}"
        if "RTb" in L:
            on2, off2 = bf2.onsets_from_ttl(L["ttl"][:, 1], L["srate"])
            line += (f"; second TTL row {len(on2)} pulses (median {np.median(off2 - on2):.2f} s); "
                     f"LAN.RTa {len(L['RTa'])}, LAN.RTb {len(L['RTb'])} events")
        print(line)
        print(f"    files: {sorted(rec['files'])}")

In [ ]:
# Per unit: laser intensities, the stored rate against the rate counted from the spike times, the
# per-unit laser and pinch onset lists, and the dataset's own response flags.
print("intensity (column 'intensity'):", collections.Counter(u["intensity"] for u in units))
print("'Int. real':", collections.Counter(round(u["int_real"], 1) for u in units))
print("pulse durations (s):", collections.Counter(tuple(np.round(u["pulse"], 2)) for u in units))
dur = np.array([lan[(a, u["rec"])]["pnts"] / lan[(a, u["rec"])]["srate"] for u, a in zip(units, unit_animal)])
rate_counted = np.array([len(u["spikes"]) for u in units]) / dur
frate = np.array([u["frate"] for u in units])
print(f"spike times in ms: range {min(u['spikes'].min() for u in units):.1f} to "
      f"{max(u['spikes'].max() for u in units):.1f}; recording lengths {sorted(set(np.round(dur)))} s")
print(f"unsorted trains {sum((np.diff(u['spikes']) < 0).any() for u in units)}, "
      f"repeated times {sum((np.diff(u['spikes']) == 0).any() for u in units)}")
print(f"stored 'f rate' / rate counted over the whole recording: median {np.median(frate / rate_counted):.3f}, "
      f"range {np.min(frate / rate_counted):.3f} to {np.max(frate / rate_counted):.3f}")
print(f"spike counts per unit: {min(len(u['spikes']) for u in units)} to {max(len(u['spikes']) for u in units)}; "
      f"units with exactly 20000 spikes: {sum(len(u['spikes']) == 20000 for u in units)}")
sig = collections.Counter((u["spikes"].size, round(u["spikes"][0], 2), round(u["spikes"][-1], 2)) for u in units)
print("spike trains deposited twice (same count, first and last time):", sum(v > 1 for v in sig.values()))

# Per-unit laser onsets against the laser TTL.
worst, n_laser = [], collections.Counter()
for u, a in zip(units, unit_animal):
    L = lan[(a, u["rec"])]
    on, _ = bf2.onsets_from_ttl(L["ttl"][:, 0], L["srate"])
    d = np.abs(on[:, None] * 1000 - u["rt_laser"][None, :]).min(axis=0)
    worst.append(d.max())
    n_laser[(a, len(on), len(u["rt_laser"]), len(u["rt_somatic"]))] += 1
print(f"\nevery per-unit laser onset ('RT laser') is within {max(worst):.1f} ms of a rising edge of the laser TTL")
print("(animal, TTL pulses, 'RT laser' onsets, 'RT somatic' onsets): units")
for k, v in sorted(n_laser.items()):
    print("  ", k, v)
# Light pulses given together with the pinch (NE52, NE58): for each recording, the laser TTL onsets
# with a pinch event (LAN.RTb) within 1 s, and whether the per-unit lists hold any of them.
print("\n(animal rec: laser TTL onsets, of them with a pinch event within 1 s and the median lag pinch - light; "
      "'RT laser' onsets, of them within 1 s of a pinch; 'RT somatic' onsets, of them within 1 s of a light onset)")
for a in ("NE52", "NE58"):
    r = [x for x in animals if x["animal"] == a][0]
    for rec in sorted(set(u["rec"] for u in r["units"])):
        u = [x for x in r["units"] if x["rec"] == rec][0]
        L = lan[(a, rec)]
        on, _ = bf2.onsets_from_ttl(L["ttl"][:, 0], L["srate"])
        pin = L["RTb"] / 1000
        lag = np.array([(pin - o)[np.argmin(np.abs(pin - o))] for o in on])
        both = np.abs(lag) < 1
        rl = sum(np.abs(pin - o).min() < 1 for o in u["rt_laser"] / 1000)
        rs = sum(np.abs(on - o).min() < 1 for o in u["rt_somatic"] / 1000)
        print(f"  {a} {rec}: {len(on)}, {both.sum()} (lag {np.median(lag[both]) * 1000:.0f} ms); "
              f"{len(u['rt_laser'])}, {rl}; {len(u['rt_somatic'])}, {rs}")

flags = np.array([u["sig_laser"] for u in units])
print("\ndataset's own light flags (column 17, 1 excited, -1 inhibited, 0 none):",
      {k: int((flags == k).sum()) for k in (1, -1, 0)}, "paper: 56, 29, 245")
for a in bf2.VP_ANIMALS:
    print("  ", a, {k: int(((flags == k) & (unit_animal == a)).sum()) for k in (1, -1, 0)})

In recordings with tail pinch (NE52, NE58) about half of the light pulses are given together with
the pinch (the pinch event falls within a few ms of the light onset), and the other pulses, and the
other pinch events, are given alone (printed above). The per-unit list `RT laser` holds only the
light-alone pulses and `RT somatic` only the pinch-alone events. The analysis uses the per-unit lists,
so light responses are measured on light-alone pulses; every TTL pulse is run as a variant.
NE48 and NE69 have one TTL row and no pinch events (printed above).

**Clock check.** The spikes and the light TTL come from the same Intan file. As a contrast, the
mean absolute log ratio of the rate in the 5 s after each onset to the 5 s before, over units, is
compared between the true onsets and the onsets moved 10 s later (into the light-off period).


In [ ]:
def logratio(sp, onsets):
    a = np.array([((sp >= o) & (sp < o + 5)).sum() for o in onsets]).sum()
    b = np.array([((sp >= o - 5) & (sp < o)).sum() for o in onsets]).sum()
    return np.log2((a + 1) / (b + 1))


rows = collections.defaultdict(lambda: [[], []])
for u, a in zip(units, unit_animal):
    sp, on = u["spikes"] / 1000, u["rt_laser"] / 1000
    rows[(a, u["rec"])][0].append(abs(logratio(sp, on)))
    rows[(a, u["rec"])][1].append(abs(logratio(sp, on + 10)))
n_larger = 0
for k, (t, s) in sorted(rows.items()):
    n_larger += np.mean(t) > np.mean(s)
    print(f"{k[0]} {k[1]}: true onsets {np.mean(t):.3f}, shifted 10 s {np.mean(s):.3f} (units {len(t)})")
print(f"recordings where the response to the true onsets is larger: {n_larger} of {len(rows)}")

In [ ]:
# Open-field archives: members, and which mice have laser timing ('ttl') and a frame-difference vector ('cambio').
for b in (beh_vp, beh_ctl):
    mice = collections.defaultdict(list)
    for name, size in b["members"]:
        if "/segm/" in name:
            continue
        mice[name.split("/")[1]].append((name.split("/")[-1], size))
    print(f"\n{b['path']}: {len(mice)} mice, {len(b['members'])} files")
    for m, files in mice.items():
        mats = {k: bf2.read_mat_any(v) for k, v in b["mats"].items() if f"/{m}/" in k}
        has = {k.split("/")[-1]: sorted(x for x in v if x in ("ttl", "cambio", "laser", "trajectories")) for k, v in mats.items()}
        timing = [k for k, v in has.items() if "ttl" in v]
        ttl_n = [np.size(mats[k2]["ttl"]) for k2 in mats if "ttl" in mats[k2]]
        print(f"  {m}: videos {[f for f, s in files if f.lower().endswith(('.avi', '.mpg'))]}, "
              f"files with laser timing {timing} (trains {ttl_n})")

# Folders of the VP open-field archive (fig5c-light-path-needed: is any session light-blocked?).
print("\nfolders of", beh_vp["path"], "outside segm/:")
for d, n in sorted(collections.Counter("/".join(name.split("/")[:-1]) for name, _ in beh_vp["members"]
                                       if "/segm/" not in name).items()):
    print(f"  {d}: {n} files")

import io, fsspec, scipy.io as sio
with fsspec.open(f"{bf2.BUCKET}/Behavioral_tests/OPEN_FIELD/result_rapidez_all.mat") as f:
    rap = sio.loadmat(io.BytesIO(f.read()))
for k, v in rap.items():
    if not k.startswith("__"):
        print(f"result_rapidez_all.mat {k}: shape {v.shape}, range {np.nanmin(v):.3g} to {np.nanmax(v):.3g}")

## fig1-bf-response-classes

> Light inhibition of somatostatin cells in vivo changes the firing of a minority of basal forebrain
> units. Of 330 units (4 animals), 29 (8.8%) decrease their activity by 49.2% and 56 (17%) increase it
> by 39.6%; 245 are unaffected.

Method from the claims file (opening comment, shared method): per unit, a 30-s window around each
light onset, 500-ms bins (250-ms variant), normalized by the spikes in the window (n1 per pulse, n2
summed), 100 shuffles of inter-spike and inter-pulse intervals, P < 0.01, readings r1 to r3.
Pulse bins are those whose centres fall in (0, 5] s (bf2.py). Seeds: this section's generator is
seeded per unit from 1 000 000 plus the unit's index.

In [ ]:
t = time.time()
KEYS = [(0.5, "n1"), (0.5, "n2"), (0.25, "n1"), (0.25, "n2")]
cls = []
for i, u in enumerate(units):
    sp, on = u["spikes"] / 1000, u["rt_laser"] / 1000
    cls.append({k: bf2.classify_unit(sp, on, k[0], k[1], seed=1_000_000 + i) for k in KEYS})
print(f"classified {len(units)} units under {len(KEYS)} bin and normalization readings in {time.time() - t:.0f} s")
print(f"units with a pulse window without spikes (n1 drops those pulses): "
      f"{sum(any(((u['spikes'] / 1000 >= o - 15.25) & (u['spikes'] / 1000 < o + 15.25)).sum() == 0 for o in u['rt_laser'] / 1000) for u in units)}")
print(f"units with an undefined normalized curve: {sum(not np.isfinite(c[k]['pulse_mean']) for c in cls for k in KEYS)}")


def classes(key, r):
    return np.array([c[key][r] for c in cls])


def pct(key):
    return np.array([c[key]["pct"] for c in cls])


def within(x, v, tol):
    return abs(x - v) <= tol + 1e-9


rows = []
for key in KEYS:
    for r in ("r1", "r2", "r3"):
        c, p = classes(key, r), pct(key)
        inh, exc, una = (c == -1).sum(), (c == 1).sum(), (c == 0).sum()
        di, de = -np.mean(p[c == -1]), np.mean(p[c == 1])
        counts_ok = within(inh, 29, 2.9) and within(exc, 56, 5.6) and within(una, 245, 24.5)
        change_ok = within(di, 49.2, 5.1) and within(de, 39.6, 7.9)
        rows.append((key, r, inh, exc, una, di, de, counts_ok, change_ok))
        print(f"{key[0] * 1000:.0f} ms {key[1]} {r}: inhibited {inh}, excited {exc}, unaffected {una}; "
              f"decrease {di:.1f}% (s.e.m. {stats.sem(-p[c == -1]):.1f}), increase {de:.1f}% "
              f"(s.e.m. {stats.sem(p[c == 1]):.1f}); counts within tolerance {counts_ok}, changes within {change_ok}")
print("\nexpected: inhibited 29 +- 2.9, excited 56 +- 5.6, unaffected 245 +- 24.5, decrease 49.2 +- 5.1, increase 39.6 +- 7.9")
print(f"across readings: inhibited {min(r[2] for r in rows)} to {max(r[2] for r in rows)}, excited "
      f"{min(r[3] for r in rows)} to {max(r[3] for r in rows)}, unaffected {min(r[4] for r in rows)} to {max(r[4] for r in rows)}, "
      f"decrease {min(r[5] for r in rows):.1f} to {max(r[5] for r in rows):.1f}%, increase {min(r[6] for r in rows):.1f} to {max(r[6] for r in rows):.1f}%")
print("readings with all three counts within tolerance:", sum(r[7] for r in rows), "of", len(rows))
print("readings with both changes within tolerance:", sum(r[8] for r in rows), "of", len(rows))

In [ ]:
# Main reading for the record: 500-ms bins (Methods), n2, r1. By animal:
key = (0.5, "n2")
c = classes(key, "r1")
for a in bf2.VP_ANIMALS:
    m = unit_animal == a
    print(f"{a}: units {m.sum()}, inhibited {(c[m] == -1).sum()}, excited {(c[m] == 1).sum()}, unaffected {(c[m] == 0).sum()}")
# Agreement with the dataset's own flags (column 17), under each reading.
for key in KEYS:
    for r in ("r1", "r2", "r3"):
        cc = classes(key, r)
        print(f"{key[0] * 1000:.0f} ms {key[1]} {r}: same class as the dataset's flag for {(cc == flags).sum()} of {len(cc)} units")
# Variant: the dataset's own classes, with the percent change measured here.
for key in KEYS:
    p = pct(key)
    print(f"variant, dataset's own classes, {key[0] * 1000:.0f} ms {key[1]}: inhibited {(flags == -1).sum()} "
          f"decrease {-np.mean(p[flags == -1]):.1f}% (s.e.m. {stats.sem(-p[flags == -1]):.1f}); excited {(flags == 1).sum()} "
          f"increase {np.mean(p[flags == 1]):.1f}% (s.e.m. {stats.sem(p[flags == 1]):.1f})")

In [ ]:
# Reading on the pulses used: the paper does not say whether light pulses given together with the
# pinch (NE52, NE58) count. Main analysis: the per-unit light-alone onsets ('RT laser'). Here: every
# light pulse of the laser TTL. Seeds: 3 000 000 plus the unit's index.
cls_all = []
for i, (u, a) in enumerate(zip(units, unit_animal)):
    L = lan[(a, u["rec"])]
    on, _ = bf2.onsets_from_ttl(L["ttl"][:, 0], L["srate"])
    cls_all.append({k: bf2.classify_unit(u["spikes"] / 1000, on, k[0], k[1], seed=3_000_000 + i) for k in KEYS})
n_counts = 0
for key in KEYS:
    for r in ("r1", "r2", "r3"):
        c = np.array([x[key][r] for x in cls_all])
        p = np.array([x[key]["pct"] for x in cls_all])
        ok = within((c == -1).sum(), 29, 2.9) and within((c == 1).sum(), 56, 5.6) and within((c == 0).sum(), 245, 24.5)
        n_counts += ok
        print(f"all TTL pulses, {key[0] * 1000:.0f} ms {key[1]} {r}: inhibited {(c == -1).sum()}, excited {(c == 1).sum()}, "
              f"unaffected {(c == 0).sum()}; decrease {-np.mean(p[c == -1]):.1f}%, increase {np.mean(p[c == 1]):.1f}%; counts within tolerance {ok}")
print(f"readings with all three counts within tolerance, all TTL pulses: {n_counts} of {len(KEYS) * 3}")

**The data set's own cross-correlograms.** Column 6 of `data_correlation.mat` (`axis/xcorr_laser`)
holds a cross-correlogram per unit, and column 18 the mean of the 100 shuffles. Reconstructing column 6
from the spike times shows the bins, the pulses and the normalization the authors used. The classes
are then computed under that convention, cross-tabulated against the data set's flags (column 17),
and compared with a sham (onsets moved 10 s later, into the light-off period) for the chance level.


In [ ]:
ax6 = units[0]["xcorr_axis"]
print(f"column 6 axis: {len(ax6)} values, {ax6[0]:.0f} to {ax6[-1]:.0f} ms, step {np.diff(ax6)[0]:.0f} ms; "
      f"same axis for every unit: {all(np.array_equal(u['xcorr_axis'], ax6) for u in units)}")
ttl_on = [bf2.onsets_from_ttl(lan[(a, u["rec"])]["ttl"][:, 0], lan[(a, u["rec"])]["srate"])[0] for u, a in zip(units, unit_animal)]
for conv in ("centred", "stored"):
    for nm in ("RT laser", "all TTL pulses"):
        n_eq, worst = 0, 0.0
        for u, on_all in zip(units, ttl_on):
            on = u["rt_laser"] / 1000 if nm == "RT laser" else on_all
            c, _ = bf2.peth_counts(u["spikes"] / 1000, on, 0.25, convention=conv)
            d = np.abs(c.sum(axis=0) / len(on) - u["xcorr"]).max()
            n_eq += d < 1e-9
            worst = max(worst, d)
        print(f"250-ms bins, {conv} edges, {nm}: column 6 equals spikes per pulse per bin for {n_eq} of {len(units)} "
              f"units (largest difference {worst:.3g} spikes per pulse)")
S18 = np.array([u["shuffle_laser"] for u in units])
print(f"column 18 (shuffle mean): {S18.shape[1]} values per unit, sums {S18.sum(axis=1).min():.6f} to "
      f"{S18.sum(axis=1).max():.6f}; interior values {S18[:, 1:-1].min():.5f} to {S18[:, 1:-1].max():.5f} "
      f"(1/120 = {1 / 120:.5f}), end values {S18[:, [0, -1]].mean():.5f}")


In [ ]:
# Classes under the stored convention (250-ms bins, stored edges, light-alone pulses, n2: the
# shuffle mean is a fraction of the window's spikes, so counts are summed over pulses, then
# normalized). Seeds: 4 000 000 plus the unit's index. Sham: onsets moved 10 s later; seeds
# 5 000 000 (stored convention) and 6 000 000 (main readings) plus the unit's index.
cls_st = [bf2.classify_unit(u["spikes"] / 1000, u["rt_laser"] / 1000, 0.25, "n2", seed=4_000_000 + i, convention="stored")
          for i, u in enumerate(units)]
sham_st = [bf2.classify_unit(u["spikes"] / 1000, u["rt_laser"] / 1000 + 10, 0.25, "n2", seed=5_000_000 + i, convention="stored")
           for i, u in enumerate(units)]
sham = [{k: bf2.classify_unit(u["spikes"] / 1000, u["rt_laser"] / 1000 + 10, k[0], k[1], seed=6_000_000 + i) for k in KEYS}
        for i, u in enumerate(units)]


def xtab(c):
    """Rows: class found here (excited, inhibited, unaffected); columns: the data set's flag."""
    return " / ".join(f"{lab} {[int(((c == k) & (flags == f)).sum()) for f in (1, -1, 0)]}"
                      for k, lab in ((1, "exc"), (-1, "inh"), (0, "unaff")))


print("cross-tabulation, class found here against the data set's flag [excited, inhibited, unaffected]:")
for r in ("r1", "r2", "r3"):
    c = np.array([x[r] for x in cls_st])
    s = np.array([x[r] for x in sham_st])
    p = np.array([x["pct"] for x in cls_st])
    ok = within((c == -1).sum(), 29, 2.9) and within((c == 1).sum(), 56, 5.6) and within((c == 0).sum(), 245, 24.5)
    print(f"stored convention {r}: inhibited {(c == -1).sum()}, excited {(c == 1).sum()}, unaffected {(c == 0).sum()} "
          f"(counts within tolerance {ok}); decrease {-np.mean(p[c == -1]):.1f}%, increase {np.mean(p[c == 1]):.1f}%; "
          f"same class as the flag {(c == flags).sum()} of 330; {xtab(c)}; sham: inhibited {(s == -1).sum()} "
          f"(of them excited by the light {((s == -1) & (c == 1)).sum()}), excited {(s == 1).sum()} "
          f"(of them inhibited by the light {((s == 1) & (c == -1)).sum()}); sham classes among units unaffected by the light: "
          f"inhibited {((s == -1) & (c == 0)).sum()}, excited {((s == 1) & (c == 0)).sum()} of {(c == 0).sum()}")
for key in KEYS:
    for r in ("r1", "r2", "r3"):
        c = classes(key, r)
        s = np.array([x[key][r] for x in sham])
        print(f"{key[0] * 1000:.0f} ms {key[1]} {r}: {xtab(c)}; sham: inhibited {(s == -1).sum()}, excited {(s == 1).sum()} "
              f"({((s != 0).sum()) / 330 * 100:.1f}% of units); among units unaffected by the light: "
              f"{((s != 0) & (c == 0)).sum()} of {(c == 0).sum()}")


In [ ]:
# Variant of the percent change (verifier, round 1): the rate during the pulse over the rate in the
# 5 s before it, spikes summed over the light-alone pulses, minus 1, times 100.
def pct_before(u):
    sp, on = u["spikes"] / 1000, u["rt_laser"] / 1000
    during = sum(((sp >= o) & (sp < o + 5)).sum() for o in on)
    before = sum(((sp >= o - 5) & (sp < o)).sum() for o in on)
    return (during / before - 1) * 100 if before else np.nan


pct_pre = np.array([pct_before(u) for u in units])
print(f"units with no spike in the 5 s before any pulse (left out): {np.isnan(pct_pre).sum()}")
for nm, c in [("500 ms n1 r1", classes((0.5, "n1"), "r1")), ("500 ms n2 r1", classes((0.5, "n2"), "r1")),
              ("stored convention r1", np.array([x["r1"] for x in cls_st])), ("data set's flags", flags)]:
    print(f"{nm}: decrease of inhibited units {-np.nanmean(pct_pre[c == -1]):.1f}% (n {(c == -1).sum()}), "
          f"increase of excited units {np.nanmean(pct_pre[c == 1]):.1f}% (n {(c == 1).sum()})")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for j, key in enumerate([(0.5, "n2"), (0.25, "n2")]):
    c = classes(key, "r1")
    for k, col, lab in ((1, "tab:red", "excited"), (-1, "k", "inhibited"), (0, "tab:blue", "unaffected")):
        cur = np.array([cls[i][key]["curve"] for i in np.flatnonzero(c == k)])
        x = cls[0][key]["centres"]
        ax[j].plot(x, np.nanmean(cur, axis=0), color=col, label=f"{lab} (n={len(cur)})")
        print(f"{key}: {lab} mean curve range {np.nanmin(np.nanmean(cur, axis=0)):.2f} to {np.nanmax(np.nanmean(cur, axis=0)):.2f}")
    ax[j].axvspan(0, 5, color="g", alpha=0.15)
    ax[j].set(xlabel="time from light onset (s)", ylabel="normalized discharge", title=f"{key[0] * 1000:.0f} ms bins, n2, r1")
ax[0].legend()
plt.tight_layout()

**Result: not reproduced.** Under none of the twelve readings (two bin sizes, n1 and n2, r1 to r3)
are the three counts within tolerance: the method finds 36 to 70 inhibited and 68 to 118 excited
units, against 29 and 56, and the decrease of inhibited units (16.2 to 39.7%) is below 49.2 - 5.1
under every reading. Counting every light pulse, including those given with the pinch, does not
bring the counts within tolerance under any reading either.

The data set's own cross-correlograms (column 6) are reproduced exactly for all 330 units with
250-ms bins, the stored edge convention and the light-alone pulses (`RT laser`), and its shuffle
means (column 18) sum to 1, as counts summed over pulses and then normalized (n2) give. Under that
convention r1 to r3 give 40 to 48 inhibited and 76 to 111 excited units, again outside tolerance.
So the authors' bins, pulses and normalization are among the readings run here, and the difference
lies in the test that turns a curve into a class.

The data set's flags (column 17) give 56 excited, 30 inhibited and 244 unaffected units, within
tolerance of the paper's counts. Under r1 and r3, 49 to 53 of the 56 flagged excited units and 22 to
25 of the 30 flagged inhibited units are found in the same class; the extra units found here are
units the flags call unaffected (under the stored convention and r1, 25 excited and 21 inhibited).
With the flags' classes the increase measured here (33.5 to 35.7%) is within tolerance and the
decrease (43.1 to 46.4%) is within it under two of four readings; measured against the 5 s before
each pulse (a variant), the flags' classes give a decrease of 51.9% and an increase of 39.1%, both
within tolerance, and the r1 classes 36.6 to 38.3% and 30.0 to 31.1%.

Sham onsets 10 s into the light-off period classify 0 to 5 of the 205 to 226 units unaffected by the
light as responsive under r1 and r3, and 32 to 38 of 142 to 190 under r2. They also classify 34 to
44 units as inhibited under r1 and r3, most of them units excited by the light (36 of 42 under the
stored convention and r1): normalizing by the 30-s window, which holds the neighbouring pulses, makes
a light-excited unit look inhibited at the sham. The sham is therefore a chance level only for the
units the light leaves unaffected.

## fig1d-excited-cells-fire-faster

> Basal forebrain units excited by the light have higher spontaneous firing rates than inhibited or
> unaffected units.

Classes from fig1-bf-response-classes under each reading. Basal rate readings from the criterion:
(a) the rate outside the light pulses over the recording, (b) the rate in the 5 s before each pulse.

In [ ]:
def basal_rates(u, a):
    L = lan[(a, u["rec"])]
    sp = u["spikes"] / 1000
    T = L["pnts"] / L["srate"]
    on, off = bf2.onsets_from_ttl(L["ttl"][:, 0], L["srate"])
    inside = sum(((sp >= x) & (sp < y)).sum() for x, y in zip(on, off))
    outside = (len(sp) - inside) / (T - (off - on).sum())
    pre = np.mean([((sp >= o - 5) & (sp < o)).sum() / 5 for o in u["rt_laser"] / 1000])
    # Outside the light and the 5 s from each pinch event (LAN.RTb, NE52 and NE58), on a 1-ms grid.
    free = np.ones(int(round(T * 1000)) + 1, bool)
    for x, y in list(zip(on, off)) + [(t / 1000, t / 1000 + 5) for t in L.get("RTb", [])]:
        free[int(x * 1000):int(y * 1000)] = False
    no_pinch = free[np.minimum((sp * 1000).astype(int), len(free) - 1)].sum() / (free.sum() / 1000)
    return outside, pre, no_pinch


B = np.array([basal_rates(u, a) for u, a in zip(units, unit_animal)])
print(f"rate outside light: median {np.median(B[:, 0]):.2f} Hz, range {B[:, 0].min():.2f} to {B[:, 0].max():.2f}; "
      f"5 s before pulses: median {np.median(B[:, 1]):.2f}, range {B[:, 1].min():.2f} to {B[:, 1].max():.2f}; "
      f"outside light and pinch: median {np.median(B[:, 2]):.2f}, range {B[:, 2].min():.2f} to {B[:, 2].max():.2f}")
n_ok = 0
for key in KEYS:
    for r in ("r1", "r2", "r3"):
        c = classes(key, r)
        for j, nm in enumerate(("outside light", "5 s before", "outside light and pinch")):
            g = {k: B[c == k, j] for k in (1, -1, 0)}
            kw = stats.kruskal(g[1], g[-1], g[0]).pvalue
            pei = stats.ranksums(g[1], g[-1]).pvalue
            peu = stats.ranksums(g[1], g[0]).pvalue
            piu = stats.ranksums(g[-1], g[0]).pvalue
            ok = kw < 0.05 and pei < 0.05 and peu < 0.05 and np.median(g[1]) > max(np.median(g[-1]), np.median(g[0]))
            n_ok += ok
            print(f"{key[0] * 1000:.0f} ms {key[1]} {r}, {nm}: median Hz excited {np.median(g[1]):.2f}, inhibited "
                  f"{np.median(g[-1]):.2f}, unaffected {np.median(g[0]):.2f}; Kruskal-Wallis p = {kw:.2g}; rank-sum "
                  f"exc-inh p = {pei:.2g}, exc-unaff p = {peu:.2g}, inh-unaff p = {piu:.2g}; holds {ok}")
print(f"readings where the criterion holds: {n_ok} of {3 * len(KEYS) * 3}")
g = {k: B[flags == k, 0] for k in (1, -1, 0)}
print(f"variant, dataset's own classes, outside light: medians {np.median(g[1]):.2f}, {np.median(g[-1]):.2f}, "
      f"{np.median(g[0]):.2f} Hz; Kruskal-Wallis p = {stats.kruskal(g[1], g[-1], g[0]).pvalue:.2g}")
c = classes((0.5, "n2"), "r1")
for a in bf2.VP_ANIMALS:
    m = unit_animal == a
    print(f"{a} (500 ms, n2, r1, outside light): median excited {np.median(B[m & (c == 1), 0]):.2f}, "
          f"others {np.median(B[m & (c != 1), 0]):.2f} Hz (excited {(m & (c == 1)).sum()}, others {(m & (c != 1)).sum()})")

**Result: reproduced.** Under all 36 combinations of class reading (12) and basal-rate reading
(outside the light, the 5 s before each pulse, outside the light and the 5 s after each pinch event),
the Kruskal-Wallis test gives p < 0.05 and excited units have a higher median rate than inhibited
and unaffected units by rank-sum tests at p < 0.05. With the data set's own classes the same holds (Kruskal-Wallis p =
3.2e-08). The classes come from fig1-bf-response-classes, whose counts are not reproduced.

## fig1d-discharge-by-class

> Normalized discharge probability during the light differs between excited, inhibited and
> unaffected basal forebrain units.

In [ ]:
n_ok = 0
for key in KEYS:
    pm = np.array([c[key]["pulse_mean"] for c in cls])
    for r in ("r1", "r2", "r3"):
        c = classes(key, r)
        g = {k: pm[c == k] for k in (1, -1, 0)}
        kw = stats.kruskal(g[1], g[-1], g[0]).pvalue
        pe_u = stats.ranksums(g[1], g[0]).pvalue
        pu_i = stats.ranksums(g[0], g[-1]).pvalue
        pe_i = stats.ranksums(g[1], g[-1]).pvalue
        order = np.median(g[1]) > np.median(g[0]) > np.median(g[-1])
        ok = kw < 0.05 and max(pe_u, pu_i, pe_i) < 0.05 and order
        n_ok += ok
        print(f"{key[0] * 1000:.0f} ms {key[1]} {r}: medians excited {np.median(g[1]):.3f}, unaffected {np.median(g[0]):.3f}, "
              f"inhibited {np.median(g[-1]):.3f}; Kruskal-Wallis p = {kw:.2g}; rank-sum exc-unaff {pe_u:.2g}, "
              f"unaff-inh {pu_i:.2g}, exc-inh {pe_i:.2g}; holds {ok}")
print(f"readings where the criterion holds: {n_ok} of {len(KEYS) * 3}")

**Result: reproduced.** Under all twelve readings the Kruskal-Wallis test and the three pairwise
rank-sum tests give p < 0.05, with excited above unaffected above inhibited. As the claims file notes,
this largely follows from the classification itself.

## fig1e-excited-slower-kinetics

> Excited basal forebrain units respond with slower kinetics than inhibited ones (time constants
> 602.5 against 263.1 ms).

Per unit, a single exponential approaching a plateau, fitted to the normalized discharge from light
onset (criterion): w1 from onset to the first extremum of the response (the first pulse bin after
which the curve stops moving away from its start, in the direction of the unit's pulse mean), w2 over
the whole pulse. The round-1 window, up to the pulse bin of largest deviation from 1, is kept as a
variant (w1max). The bin centred on onset gives the value at t = 0. A w1 window of fewer than three
bins cannot fit the three parameters and is dropped. The time constant is bounded to 1 ms to 20 s, a
standard choice; fits at the upper bound are counted. A synthetic check of the fit comes first.


In [ ]:
# Synthetic check of fit_tau (seeded): 40 Poisson units per condition at 10 Hz whose rate during
# each 5-s pulse approaches 14 Hz (excited) or 6 Hz (inhibited) with a known time constant, 28
# pulses every 20 s. Printed: the median time constant recovered, the fits dropped and at the bound,
# and the rank-sum test of excited against inhibited units with the same true time constant (a
# difference here would come from the method, not the kinetics).
rng = np.random.default_rng(7_000_000)
syn = collections.defaultdict(list)
for tau_true in (0.26, 0.6):
    for gain in (1.4, 0.6):
        for _ in range(40):
            t = np.arange(0, 600, 0.001)
            ph = (t - 5) % 20
            rate = np.where(ph < 5, 10 * (1 + (gain - 1) * (1 - np.exp(-ph / tau_true))), 10.0)
            sp = t[rng.random(t.size) < rate * 0.001]
            for k in (0.5, 0.25):
                cnt, cen = bf2.peth_counts(sp, np.arange(25, 600 - 20, 20.0), k)
                for nn in ("n1", "n2"):
                    cur = bf2.normalize(cnt, nn)
                    for w in ("w1", "w1max", "w2"):
                        syn[(tau_true, gain, k, nn, w)].append(bf2.fit_tau(cur, cen, w))
for tau_true in (0.26, 0.6):
    for k in (0.5, 0.25):
        for nn in ("n1", "n2"):
            for w in ("w1", "w1max", "w2"):
                e, i = np.array(syn[(tau_true, 1.4, k, nn, w)]), np.array(syn[(tau_true, 0.6, k, nn, w)])
                ef, if_ = e[np.isfinite(e)], i[np.isfinite(i)]
                print(f"true {tau_true * 1000:.0f} ms, {k * 1000:.0f} ms {nn} {w}: excited median {np.median(ef):.0f} ms "
                      f"({40 - len(ef)} dropped, {(ef > 19900).sum()} at the bound), inhibited median {np.median(if_):.0f} ms "
                      f"({40 - len(if_)} dropped, {(if_ > 19900).sum()} at the bound); rank-sum p = {stats.ranksums(ef, if_).pvalue:.2g}")


In [ ]:
WINDOWS = ("w1", "w1max", "w2")
TAU = {(key, w): np.array([bf2.fit_tau(c[key]["curve"], c[key]["centres"], w) for c in cls]) for key in KEYS for w in WINDOWS}
n_ok, n_main = 0, 0
for key in KEYS:
    for w in WINDOWS:
        tau = TAU[(key, w)]
        for r in ("r1", "r2", "r3"):
            c = classes(key, r)
            e, i = tau[c == 1], tau[c == -1]
            ef, if_ = e[np.isfinite(e)], i[np.isfinite(i)]
            p = stats.ranksums(ef, if_).pvalue
            ok = p < 0.05 and np.median(ef) > np.median(if_)
            n_ok += ok
            n_main += ok and w != "w1max"
            print(f"{key[0] * 1000:.0f} ms {key[1]} {r} {w}: excited mean {ef.mean():.0f} ms (median {np.median(ef):.0f}, "
                  f"{len(ef)} fitted of {len(e)}, {(ef > 19900).sum()} at the bound), inhibited mean {if_.mean():.0f} ms "
                  f"(median {np.median(if_):.0f}, {len(if_)} of {len(i)}, {(if_ > 19900).sum()} at the bound); "
                  f"rank-sum p = {p:.3g}; test holds {ok}; means within tolerance "
                  f"{within(ef.mean(), 602.5, 68.6) and within(if_.mean(), 263.1, 48.7)}")
print(f"readings where the rank-sum test holds with excited slower: w1 and w2 {n_main} of {len(KEYS) * 2 * 3}, "
      f"all three windows {n_ok} of {len(KEYS) * 3 * 3}")


In [ ]:
# Variant: the data set's own classes (column 17), and the distribution of the time constants per class.
EDGES = [0, 100, 200, 400, 800, 1600, 3200, 6400, 12800, 19900, 20001]
for key in KEYS:
    for w in WINDOWS:
        tau = TAU[(key, w)]
        e, i = tau[flags == 1], tau[flags == -1]
        ef, if_ = e[np.isfinite(e)], i[np.isfinite(i)]
        p = stats.ranksums(ef, if_).pvalue
        print(f"data set's classes, {key[0] * 1000:.0f} ms {key[1]} {w}: excited mean {ef.mean():.0f} ms (median {np.median(ef):.0f}, "
              f"{len(ef)} of {len(e)}, {(ef > 19900).sum()} at the bound), inhibited mean {if_.mean():.0f} ms (median "
              f"{np.median(if_):.0f}, {len(if_)} of {len(i)}, {(if_ > 19900).sum()} at the bound); rank-sum p = {p:.3g}, "
              f"excited slower {np.median(ef) > np.median(if_)}")
print(f"\nhistograms of the time constant (ms), bin edges {EDGES[:-1]} and the bound:")
for key in KEYS:
    for w in WINDOWS:
        tau = TAU[(key, w)]
        for nm, c in (("r1", classes(key, "r1")), ("data set's classes", flags)):
            h = {lab: np.histogram(tau[(c == k) & np.isfinite(tau)], EDGES)[0].tolist() for k, lab in ((1, "excited"), (-1, "inhibited"))}
            print(f"{key[0] * 1000:.0f} ms {key[1]} {w}, {nm}: excited {h['excited']}, inhibited {h['inhibited']}")


**Result: partially reproduced.** The criterion is met in part under one reading: with w1 (onset to
the first extremum), 250-ms bins and n1, the rank-sum test finds excited units slower than inhibited
ones under r1, r2 and r3 (p = 0.0008, 0.0013 and 5.8e-05), while the mean time constants (5669 to
15766 ms) are far outside tolerance. Under the other 21 readings with w1 or w2 the test does not hold
(p = 0.051 to 0.93), nor under any of the 12 with the round-1 window w1max. The result is fragile:
under that reading 37 to 51 excited units have fits at the 20-s bound and 22 to 42 excited and 21 to
35 inhibited units are dropped because the first extremum falls in the first pulse bin. In the
synthetic check w1 does not recover the time constant (medians 619 to 20000 ms for true 260 and 600
ms) and finds no difference between excited and inhibited units with the same time constant (p = 0.13
to 0.77), so the w1 difference is not an artefact of the window's direction; w2 recovers 221 to 289
ms for 260 ms and 424 to 542 ms for 600 ms. With the data set's own classes, w2 finds excited units slower under all four bin and
normalization readings (p = 0.0026 to 0.043; medians 531 to 728 ms against 212 to 310 ms), but the
means are again outside tolerance (2110 to 5392 ms).

## supp4-tail-pinch-response-patterns

> Of 44 pinch-inhibited units 10 are excited, 7 inhibited and 27 unaffected by light; of 149
> pinch-activated units 24, 11 and 114; and of 18 light-inhibited units 6 are excited, 7 inhibited
> and 5 unaffected by the pinch.

Pinch responses are classified by the same method around the per-unit pinch onsets (`RT somatic`),
with the pinch window 2 s (the files' pinch duration, Methods "1 to 2 s") or 5 s (legend), both read.
Seeds: 2 000 000 plus the unit's index.

In [ ]:
has_pinch = np.array([len(u["rt_somatic"]) > 0 for u in units])
print("animals with pinch timing in their recordings (a second LAN.ttl row or LAN.RTb):",
      sorted({a for (a, r), L in lan.items() if L["ttl"].shape[1] > 1 or "RTb" in L}))
idx = np.flatnonzero(has_pinch)
print(f"units with pinch onsets: {has_pinch.sum()}, by animal {dict(collections.Counter(unit_animal[has_pinch]))}; "
      f"pinch onsets per unit {sorted(set(len(units[i]['rt_somatic']) for i in idx))}")
sflag = np.array([u["sig_somatic"] if u["sig_somatic"] is not None else 9 for u in units])
print("dataset's own pinch flags (column 19) on units with pinch onsets:", dict(collections.Counter(sflag[has_pinch])),
      "; on units without:", dict(collections.Counter(sflag[~has_pinch])), "(9: empty)")


def tally(pc, lc):
    return (f"pinch-inhibited {(pc == -1).sum()} (light e/i/u {((pc == -1) & (lc == 1)).sum()}/{((pc == -1) & (lc == -1)).sum()}/"
            f"{((pc == -1) & (lc == 0)).sum()}), pinch-activated {(pc == 1).sum()} ({((pc == 1) & (lc == 1)).sum()}/"
            f"{((pc == 1) & (lc == -1)).sum()}/{((pc == 1) & (lc == 0)).sum()}), light-inhibited {(lc == -1).sum()} "
            f"(pinch e/i/u {((lc == -1) & (pc == 1)).sum()}/{((lc == -1) & (pc == -1)).sum()}/{((lc == -1) & (pc == 0)).sum()})")


n_groups = 0
for w in (2.0, 5.0):
    for key in KEYS:
        pin = [bf2.classify_unit(units[i]["spikes"] / 1000, units[i]["rt_somatic"] / 1000, key[0], key[1],
                                 seed=2_000_000 + i, pulse=w) for i in idx]
        for r in ("r1", "r2", "r3"):
            pc = np.array([x[r] for x in pin])
            lc = classes(key, r)[idx]
            ok = within((pc == -1).sum(), 44, 4.4) and within((pc == 1).sum(), 149, 14.9) and within((lc == -1).sum(), 18, 1.8)
            n_groups += ok
            print(f"window {w:.0f} s, {key[0] * 1000:.0f} ms {key[1]} {r}: {tally(pc, lc)}; group sizes within tolerance {ok}")
print(f"\nexpected: pinch-inhibited 44 (10/7/27), pinch-activated 149 (24/11/114), light-inhibited 18 (6/7/5)")
print(f"readings with the three group sizes within tolerance: {n_groups} of {2 * len(KEYS) * 3}")
print("variant, dataset's own flags on units with pinch onsets:", tally(sflag[has_pinch], flags[has_pinch]))
print("variant, dataset's own flags on all units:", tally(np.where(sflag == 9, 0, sflag), flags))

**Result: not reproduced.** Pinch timing is in the recordings of two animals (NE52 and NE58, 242
units); NE48 and NE69 have one TTL row and no pinch events, while the legend gives four animals for
panels C and D. So the counts of panels C and D (pinch-inhibited 10 to 50 against 44,
pinch-activated 6 to 62 against 149) cover only two of the four animals and cannot by themselves
decide the outcome. Panel E, which the legend gives for two animals and the files allow, is not
matched under any of the 24 readings: light-inhibited units tested with the pinch number 25 to 47
against 18, outside the tolerance of 1.8, and so the criterion fails whatever panels C and D would
give. The data set's own flags give 23 light-inhibited units among the units with pinch onsets and 30
over all units, also outside tolerance; its pinch flags over all units, including the 88 units of
NE48 and NE69 whose files hold no pinch timing, give 45 pinch-inhibited units (10, 7 and 28 by light
class) against the legend's 44 (10, 7, 27), but 61 pinch-activated units against 149.

## Blocked claims

Every other claim rests on data the data set does not hold, or holds without the identities needed.
The listings above show its contents: basal forebrain (ventral pallidum) optrode recordings, medial
septum and hippocampus recordings, and open-field and Y-maze videos.

- supp1 (histology), supp2 and supp3 (in vitro whole-cell recordings): no histology or slice data.
- fig2c, supp6, fig2d, supp7c, fig2e, fig3a, fig3b, supp9-layers, supp9-fast-spiking-cells,
  supp9-interneurons-stronger: prefrontal unit recordings; none are in the data set (the descriptor
  names only basal forebrain, medial septum and hippocampus recordings).
- fig4bcd, supp10, fig4g, supp11a, fig4h, suppt2: prefrontal LFP; not in the data set.
- fig5c: sessions with the light path blocked. Every folder of the VP open-field archive outside
  `segm/` is a `Laser_ON` or `LASER_ON` folder (printed in the data check), and no other file, nor the
  descriptor, mentions a light-blocked session.
- fig5d and supp14: the frame-difference vector and the laser train onsets are stored (`cambio`,
  `ttl`) for 3 of the 5 VP mice and for none of the 9 control mice, whose folders hold only an MPEG
  video and tracking output, and the paper's 5 NpHR- mice cannot be told apart from the 9 controls.
  `result_rapidez_all.mat` holds per-train traces for `halo_cre` (90 trains) and `control` (91 trains),
  read by `openfield_analysis.m`, a script the descriptor gives for a later paper. The script shows
  the traces span 60 * ceil(frame_rate) frames either side of train onset and takes a baseline from
  -20 to 0 s; it does not say what the quantity is (the name suggests speed) or which mice the trains
  come from. The cell below runs the criterion's steps on these traces as a substitute, which decides
  nothing: they are not the frame-difference estimator, are not z-scored per session, and carry no
  mouse identity.

In [ ]:
# result_rapidez_all.mat and the script that reads it, openfield_analysis.m (in the data set, written
# for a later paper, per the descriptor). A substitute test, which decides nothing: the criterion's
# estimator is the frame difference z-scored over a session, and these traces carry no mouse identity.
with fsspec.open(f"{bf2.BUCKET}/Behavioral_tests/OPEN_FIELD/openfield_analysis.m", "rt") as f:
    script = f.read().splitlines()
print("openfield_analysis.m, first 20 lines:")
print("\n".join(script[:20]))
fr = float(np.squeeze(rap["frame_rate"]))
tt = np.arange(-int(60 * np.ceil(fr)), int(60 * np.ceil(fr)) + 1) / fr
print(f"\nframes per trace {rap['rap_laser_halo_cre'].shape[0]}, time axis {tt[0]:.1f} to {tt[-1]:.1f} s (60 * ceil(frame_rate) frames each side)")


def substitute(X, mu, sd):
    """The criterion's steps on a speed trace: z-score, absolute value, 50-frame moving average;
    quiescent when below 2 z for the 40 s before onset; latency from 20 s after onset to the first
    crossing of 2 z up to 60 s, else 40 s; amplitude read as the mean, the maximum, and the mean of the
    part above 2 z (0 when none) from 20 to 60 s."""
    z = np.abs((X - mu) / sd)
    z = np.array([np.convolve(c, np.ones(50) / 50, mode="same") for c in z.T]).T
    quiet = (z[(tt >= -40) & (tt < 0)] < 2).all(axis=0)
    win = z[(tt >= 20) & (tt <= 60)]
    lat = np.array([np.flatnonzero(c >= 2)[0] / fr if (c >= 2).any() else 40.0 for c in win.T])
    amp = np.array([[c.mean(), c.max(), c[c > 2].mean() if (c > 2).any() else 0.0] for c in win.T])
    return quiet, lat, amp


H, Cc = rap["rap_laser_halo_cre"], rap["rap_laser_control"]
print(f"NpHR+ ('halo_cre') trains {H.shape[1]}, control trains {Cc.shape[1]} (paper: 60 + 26 = 86 NpHR+ and 49 + 34 = 83 NpHR- trains)")
both = np.r_[H.ravel(), Cc.ravel()]
for nm, (mh, sh, mc, sc) in (("z over each group's traces", (H.mean(), H.std(), Cc.mean(), Cc.std())),
                             ("z over both groups' traces", (both.mean(), both.std(), both.mean(), both.std()))):
    qh, lh, ah = substitute(H, mh, sh)
    qc, lc, ac = substitute(Cc, mc, sc)
    print(f"\n{nm}: quiescent trains NpHR+ {qh.sum()} of {len(qh)}, control {qc.sum()} of {len(qc)} (paper 60 and 49)")
    print(f"  quiescent, latency: NpHR+ {lh[qh].mean():.1f} s, control {lc[qc].mean():.1f} s (paper 8.3 and 27.2); "
          f"t-test p = {stats.ttest_ind(lh[qh], lc[qc]).pvalue:.2g}")
    for j, a in enumerate(("mean", "maximum", "mean above 2 z")):
        print(f"  quiescent, amplitude ({a}): NpHR+ {ah[qh, j].mean():.2f}, control {ac[qc, j].mean():.2f} (paper 1.92 and 0.97); "
              f"t-test p = {stats.ttest_ind(ah[qh, j], ac[qc, j]).pvalue:.2g}")
    print(f"  moving ({(~qh).sum()} and {(~qc).sum()} trains; paper 26 and 34): latency rank-sum p = "
          f"{stats.ranksums(lh[~qh], lc[~qc]).pvalue:.2g}, amplitude (mean) p = {stats.ranksums(ah[~qh, 0], ac[~qc, 0]).pvalue:.2g}")
pre_h, pre_c = H[(tt > -20) & (tt < 0)].mean(axis=0), Cc[(tt > -20) & (tt < 0)].mean(axis=0)
print(f"\nthe script's own selection (mean trace value from -20 to 0 s below 1): NpHR+ {(pre_h < 1).sum()}, control {(pre_c < 1).sum()} trains; "
      f"mean value 0 to 20 s against -20 to 0 s on those trains: NpHR+ {H[(tt >= 0) & (tt < 20)][:, pre_h < 1].mean():.3f} against "
      f"{pre_h[pre_h < 1].mean():.3f}, control {Cc[(tt >= 0) & (tt < 20)][:, pre_c < 1].mean():.3f} against {pre_c[pre_c < 1].mean():.3f}")


The substitute finds no difference between the groups: of the quiescent trains (81 of 90 NpHR+
and 84 or 86 of 91 control, against the paper's 60 and 49), latencies are 36.5 s against 36.3 to
36.9 s (t-test p = 0.78 to 0.89) and no amplitude reading differs (p = 0.14 to 0.75). The claims
stay blocked; this is reported so that the data set's per-train traces are not left unexamined.


## Summary

| Claim | Outcome |
|---|---|
| fig1-bf-response-classes | Not reproduced |
| fig1d-excited-cells-fire-faster | Reproduced |
| fig1d-discharge-by-class | Reproduced |
| fig1e-excited-slower-kinetics | Partially reproduced (one reading: w1, 250-ms bins, n1) |
| supp4-tail-pinch-response-patterns | Not reproduced |
| supp1, supp2, supp3 | Blocked: data missing |
| 16 prefrontal claims (Figures 2 to 4 and Supplementary Figures 6 to 11, Table 2) | Blocked: data missing |
| fig5c-light-path-needed | Blocked: data missing |
| fig5d-nphr-locomotion, supp14-moving-no-difference | Blocked: metadata insufficient |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

Differences from the paper's method: the response classification follows the readings the claims
file names, since the paper's own procedure is not given in full; the paper's analysis code is not
public. The data set's own cross-correlograms show the authors' bins (250 ms), pulses (light alone)
and normalization (n2), which are among the readings run here; its per-unit flags match the paper's
counts, so the test that the authors applied to the curves differs from r1, r2 and r3.

What was found about the data set: recordings last 900 s (600 s in NE69), not the 10 min of the
descriptor; light pulses come every 20, 25 or 30 s, not 15 s off; about half the light pulses of
NE52 and NE58 are given together with the pinch, which the paper does not mention; the pinch TTL
row of NE52 holds hundreds of short pulses, and the pinch events are in LAN.RTb; pinch timing exists
for 2 of the 4 animals; the data set holds no prefrontal data, although the paper's main findings
are prefrontal; control open-field videos carry no laser timing, and the per-train traces of
`result_rapidez_all.mat` carry no mouse identity or description.


In [ ]:
print(f"total run time {(time.time() - T0) / 60:.1f} min; peak memory of this process "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")